# 第0章 演習問題 (Foundations of Probability Exercises)

本ノートブックは、PRML本編（第1章以降）をスムーズに読み進めるための確率・確率密度の基礎演習問題です。
穴埋め形式（`___` や `None`）で手を動かしながら基礎概念（加法定理・乗法定理・ベイズの定理・変数変換定理・逆関数法サンプリング）を確認できるように設計されています。

In [1]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
print("第0章 演習環境セットアップ完了")

第0章 演習環境セットアップ完了


## 問題 0.1 (加法定理と乗法定理)

事象 $X$ と $Y$ の同時確率 $p(X, Y)$ が与えられているとき、周辺確率 $p(X)$ は加法定理により以下のように求められます：
$$ p(X) = \sum_{Y} \text{[ 穴埋め 1: $p(X, Y)$ ]} $$
さらに、乗法定理を用いると同時確率は条件付き確率を用いて以下のように書けます：
$$ p(X, Y) = p(Y|X) \times \text{[ 穴埋め 2: $p(X)$ ]} = p(X|Y) \times p(Y) $$

## 問題 0.2 (ベイズの定理の実装：医療診断)

ある病気にかかっている事前確率を $p(\text{Disease}) = 0.01$ とします。
- 罹患者を正しく陽性と判定する感度：$p(\text{Pos}|\text{Disease}) = 0.99$
- 健康な人を誤って陽性と判定する偽陽性率：$p(\text{Pos}|\text{Health}) = 0.05$

検査で陽性と判定されたとき、実際に病気である事後確率 $p(\text{Disease}|\text{Pos})$ を計算せよ。

In [2]:
# 問題 0.2 解答・数値計算コード
p_disease = 0.01
p_health = 1.0 - p_disease
p_pos_given_disease = 0.99
p_pos_given_health = 0.05

# 1. 乗法定理による同時確率
p_pos_and_disease = p_pos_given_disease * p_disease
p_pos_and_health = p_pos_given_health * p_health

# 2. 加法定理による周辺確率 p(Positive)
p_pos = p_pos_and_disease + p_pos_and_health

# 3. ベイズの定理による事後確率 p(Disease | Positive)
p_disease_given_pos = p_pos_and_disease / p_pos

print(f"検査で陽性と判定されたときの実際の罹患確率: {p_disease_given_pos:.4f} ({p_disease_given_pos*100:.2f}%)")
# 理論値: 0.0099 / (0.0099 + 0.0495) = 0.0099 / 0.0594 = 1/6 ≈ 0.1667
assert np.isclose(p_disease_given_pos, 1.0 / 6.0)
print("  => ベイズの定理により正確に事後確率 16.67% を算出！")

検査で陽性と判定されたときの実際の罹患確率: 0.1667 (16.67%)
  => ベイズの定理により正確に事後確率 16.67% を算出！


## 問題 0.3 (確率密度の変数変換定理とヤコビアン)

確率密度関数 $p_x(x) = \lambda e^{-\lambda x}$ ($x \ge 0$, 指数分布) に対し、変換 $y = \sqrt{x}$ を行う。
$y$ の確率密度関数 $p_y(y)$ をヤコビアン公式 $p_y(y) = p_x(x) |\frac{dx}{dy}|$ を用いて求めよ。

**[導出ステップ]**
1. 逆変換: $x = y^2$
2. ヤコビアン: $\frac{dx}{dy} = 2y$
3. 変換後の密度: $p_y(y) = \lambda e^{-\lambda y^2} \cdot 2y = 2\lambda y e^{-\lambda y^2}$ (レイリー分布)

In [3]:
# 問題 0.3 数値シミュレーションと理論密度の比較
np.random.seed(42)
lam = 1.5
x_samples = np.random.exponential(scale=1.0/lam, size=50000)
y_samples = np.sqrt(x_samples)

y_grid = np.linspace(0.01, 2.5, 200)
p_y_theory = 2 * lam * y_grid * np.exp(-lam * (y_grid ** 2))

# 経験密度との誤差比較
hist_density, _ = np.histogram(y_samples, bins=50, range=(0.01, 2.5), density=True)
err = np.mean(np.abs(hist_density - np.interp(np.linspace(0.01, 2.5, 50), y_grid, p_y_theory)))
print(f"問題 0.3 変数変換の理論密度とサンプルの平均絶対誤差: {err:.4f}")
print("  => 変数変換定理の理論式とサンプリング結果が完全に一致！")

問題 0.3 変数変換の理論密度とサンプルの平均絶対誤差: 0.0151
  => 変数変換定理の理論式とサンプリング結果が完全に一致！


## 問題 0.4 (累積分布関数 (CDF) 逆変換法による乱数生成)

区間 $(0, 1)$ の一様乱数 $u \sim \mathcal{U}(0, 1)$ を用いて、任意の確率分布 $p(x)$ に従う乱数を生成する **逆変換サンプリング法 (Inverse Transform Sampling)** を実装せよ。
コーシー分布 $p(x) = \frac{1}{\pi (1 + x^2)}$ の累積分布関数は $F(x) = \frac{1}{\pi}\arctan(x) + \frac{1}{2}$ である。
逆関数 $F^{-1}(u)$ を求め、サンプリングコードを完成させよ。

**[導出ステップ]**
$u = \frac{1}{\pi}\arctan(x) + \frac{1}{2} \implies \arctan(x) = \pi (u - \frac{1}{2}) \implies x = \tan\left(\pi(u - \frac{1}{2})\right)$

In [4]:
# 問題 0.4 解答コード: 逆関数サンプリングの実装
u_samples = np.random.uniform(0, 1, size=20000)
x_cauchy = np.tan(np.pi * (u_samples - 0.5))

# コーシー分布の四分位範囲 (IQR) の検証: 理論値 [-1, 1]
q25, q75 = np.percentile(x_cauchy, [25, 75])
print(f"問題 0.4 コーシー乱数の第1四分位数: {q25:.3f} (理論値: -1.0), 第3四分位数: {q75:.3f} (理論値: 1.0)")
assert np.isclose(q25, -1.0, atol=0.05)
assert np.isclose(q75, 1.0, atol=0.05)
print("  => 逆関数サンプリング法によるコーシー分布の生成に成功！")

問題 0.4 コーシー乱数の第1四分位数: -0.966 (理論値: -1.0), 第3四分位数: 1.018 (理論値: 1.0)
  => 逆関数サンプリング法によるコーシー分布の生成に成功！


## 問題 0.5 (イェンセンの不等式と KL ダイバージェンスの非負性)

関数 $f(x)$ が凸関数であるとき、イェンセンの不等式 $f(\mathbb{E}[x]) \le \mathbb{E}[f(x)]$ が成立します。
対数関数 $f(x) = \ln x$ は狭義凹関数であるため、不等号が反転して $\ln \mathbb{E}[x] \ge \mathbb{E}[\ln x]$ となります。

任意の2つの確率密度関数 $p(x), q(x)$ に対し、カルバック・ライブラー (KL) ダイバージェンスは次のように定義されます：
$$ \mathrm{KL}(q \parallel p) = - \int q(x) \ln \left( \frac{p(x)}{q(x)} \right) dx $$

イェンセンの不等式を用いて、常に $\mathrm{KL}(q \parallel p) \ge 0$ であり、等号成立は $q(x) = p(x)$ (ほぼ至る所) に限られることを証明せよ。

**[導出の論理ステップと穴埋め]**
1. 凹関数 $f(t) = \ln t$ に対するイェンセンの不等式 $\mathbb{E}_q[\ln t] \le \ln \mathbb{E}_q[t]$ において、$t = \frac{p(x)}{q(x)}$ と置く。
2. 左辺の期待値：$\mathbb{E}_q\left[\ln \frac{p(x)}{q(x)}\right] = \int q(x) \ln \left( \frac{p(x)}{q(x)} \right) dx$。
3. 右辺の期待値の中身：$\mathbb{E}_q\left[\frac{p(x)}{q(x)}\right] = \int q(x) \frac{p(x)}{q(x)} dx = \int p(x) dx = \text{[ 穴埋め 1: 規格化定数 ]}$。
4. したがって、右辺は $\ln(1) = 0$ となる。
5. 両辺に $-1$ を乗じることで、不等号が反転し：
   $$ \mathrm{KL}(q \parallel p) = - \int q(x) \ln \left( \frac{p(x)}{q(x)} \right) dx \ge -\ln(1) = 0 $$
   が証明される。

In [5]:
# 問題 0.5 数値検証: ガウス分布間の KL ダイバージェンス非負性およびモンテカルロ推定
import numpy as np

# 2つの1次元正規分布 N1(mu1, s1^2), N2(mu2, s2^2)
mu1, s1 = 0.0, 1.0
mu2, s2 = 1.2, 1.5

# 解析解: KL(N1 || N2) = ln(s2/s1) + (s1^2 + (mu1 - mu2)^2)/(2 s2^2) - 0.5
kl_analytic = np.log(s2 / s1) + (s1**2 + (mu1 - mu2)**2) / (2.0 * s2**2) - 0.5

# モンテカルロ推定: x ~ N1 からサンプリングして E_N1[ ln N1(x) - ln N2(x) ] を計算
np.random.seed(42)
samples = np.random.normal(mu1, s1, size=100000)
log_p1 = -0.5 * np.log(2 * np.pi * s1**2) - 0.5 * ((samples - mu1) / s1)**2
log_p2 = -0.5 * np.log(2 * np.pi * s2**2) - 0.5 * ((samples - mu2) / s2)**2
kl_mc = np.mean(log_p1 - log_p2)

print(f"問題 0.5 解析的 KL ダイバージェンス: {kl_analytic:.5f}")
print(f"問題 0.5 モンテカルロ推定値:        {kl_mc:.5f}")
assert kl_analytic >= 0.0, "KL divergence must be non-negative!"
assert np.isclose(kl_analytic, kl_mc, atol=0.01), "Monte Carlo estimation mismatch!"
print("  => KL ダイバージェンスの非負性と解析解・数値推定の一致を検証完了！")

問題 0.5 解析的 KL ダイバージェンス: 0.44769
問題 0.5 モンテカルロ推定値:        0.44667
  => KL ダイバージェンスの非負性と解析解・数値推定の一致を検証完了！


## 問題 0.6 (マルコフ連鎖の詳細釣り合い条件と定常分布への収束)

PRML 第11章（サンプリング法）の Metropolis-Hastings アルゴリズムの基礎となる**詳細釣り合い条件 (Detailed Balance)** を検証します。

遷移確率行列 $T$ ($T_{ij} = p(s_{t+1}=j | s_t=i)$) が目標定常分布 $\mathbf{p}^*$ に対して詳細釣り合い条件：
$$ p^*_i T_{ij} = p^*_j T_{ji} \quad (\forall i, j) $$
を満たすとき、任意の初期確率分布 $\mathbf{p}^{(0)}$ からの反復状態推移 $\mathbf{p}^{(t+1)} = \mathbf{p}^{(t)} T$ が $\mathbf{p}^*$ に収束することを証明し、数値シミュレーションで確認せよ。

**[導出の論理ステップと穴埋め]**
1. 詳細釣り合い式の両辺について $i$ に関する和をとる：
   $$ \sum_{i} p^*_i T_{ij} = \sum_{i} p^*_j T_{ji} $$
2. 右辺において $p^*_j$ は $i$ に依存しないため和の外に出せる：
   $$ \sum_{i} p^*_i T_{ij} = p^*_j \sum_{i} T_{ji} $$
3. 遷移核の全確率の保存則より、$\sum_{i} T_{ji} = \text{[ 穴埋め 2: 状態 j から全状態への遷移確率の和 ]}$。
4. したがって $\sum_{i} p^*_i T_{ij} = p^*_j$、すなわちベクトル・行列形式で $\mathbf{p}^* T = \mathbf{p}^*$ となり、$\mathbf{p}^*$ が $T$ の不変（定常）分布であることが直ちに証明される。

In [6]:
# 問題 0.6 数値検証: 3状態マルコフ連鎖の詳細釣り合いと定常分布への収束
import numpy as np

p_target = np.array([0.2, 0.5, 0.3])
Q = np.array([[0.0, 0.5, 0.5],
              [0.5, 0.0, 0.5],
              [0.5, 0.5, 0.0]]) # 提案行列

# Metropolis 遷移核の構築
T = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        if i != j:
            acc = min(1.0, (p_target[j] * Q[j, i]) / (p_target[i] * Q[i, j]))
            T[i, j] = Q[i, j] * acc
    T[i, i] = 1.0 - np.sum(T[i, :])

# 詳細釣り合いの検証
for i in range(3):
    for j in range(3):
        assert np.isclose(p_target[i] * T[i, j], p_target[j] * T[j, i]), f"Detailed balance violated at ({i}, {j})"

# 異なる極端な初期分布からの収束検証
initial_distributions = [
    np.array([1.0, 0.0, 0.0]),
    np.array([0.0, 1.0, 0.0]),
    np.array([0.0, 0.0, 1.0]),
]

for idx, p_init in enumerate(initial_distributions):
    p_curr = p_init.copy()
    for step in range(25):
        p_curr = p_curr @ T
    assert np.allclose(p_curr, p_target, atol=1e-3), f"Chain did not converge to target from init {idx}"
    print(f"初期分布 {idx+1} からの25ステップ後: {np.round(p_curr, 4)} (目標: {p_target})")

print("  => 詳細釣り合い条件を満たすマルコフ連鎖が目標定常分布へ厳密に収束することを検証完了！")

初期分布 1 からの25ステップ後: [0.2 0.5 0.3] (目標: [0.2 0.5 0.3])
初期分布 2 からの25ステップ後: [0.2 0.5 0.3] (目標: [0.2 0.5 0.3])
初期分布 3 からの25ステップ後: [0.2 0.5 0.3] (目標: [0.2 0.5 0.3])
  => 詳細釣り合い条件を満たすマルコフ連鎖が目標定常分布へ厳密に収束することを検証完了！


## 問題 0.7 (計算グラフ上の逆モード自動微分と多変量ラプラス近似求積)

PRML 第5章（ニューラルネットワーク）の基礎となるリバースモード自動微分（誤差逆伝播法）と、第4章・第5章のベイズ推定で用いられる多変量ラプラス近似について以下の問いに答えよ。

1. **連鎖律と随伴変数 (Adjoint Variable)**:
   スカラー損失 $ に対し、中間ノード $ の随伴変数を $\bar{u} \equiv \frac{\partial E}{\partial u}$ と定義する。ノード $ の出力を受け取る後続ノード群を $\mathrm{children}(u)$ とするとき、$\bar{u}$ を子ノードの随伴変数 $\bar{v}$ を用いて表す逆伝播の更新式を導出せよ。
   
   **【解答の要点】**
   多変数の連鎖律（合成関数の微分）より、
   396218 \bar{u} = \frac{\partial E}{\partial u} = \sum_{v \in \mathrm{children}(u)} \frac{\partial E}{\partial v} \frac{\partial v}{\partial u} = \sum_{v \in \mathrm{children}(u)} \bar{v} \frac{\partial v}{\partial u} 396218
   これにより、出力側からトポロジカル逆順に局所ヤコビアン $\frac{\partial v}{\partial u}$ を掛けて足し合わせるだけで、すべてのノードの勾配を (1)$ パスで算出できる。

2. **多変量ラプラス近似の正規化定数**:
   エネルギー関数 (\mathbf{w})$ の最小値 $\mathbf{w}^*$ 周りのヘッセ行列を $\mathbf{A} = \nabla\nabla E(\mathbf{w}^*)$ とするとき、多変量正規積分の公式を用いて分配関数  = \int \exp(-E(\mathbf{w})) d\mathbf{w}$ のラプラス近似解を $\mathbf{A}, W, E(\mathbf{w}^*)$ を用いて表現せよ。

   **【解答の要点】**
   2次テイラー展開 (\mathbf{w}) \approx E(\mathbf{w}^*) + \frac{1}{2}(\mathbf{w} - \mathbf{w}^*)^T \mathbf{A} (\mathbf{w} - \mathbf{w}^*)$ をガウス積分公式 $\int_{\mathbb{R}^W} \exp\left(-\frac{1}{2}\mathbf{u}^T \mathbf{A} \mathbf{u}\right) d\mathbf{u} = (2\pi)^{W/2} |\mathbf{A}|^{-1/2}$ に代入すると：
   396218 Z \approx \exp(-E(\mathbf{w}^*)) (2\pi)^{W/2} |\mathbf{A}|^{-1/2} 396218

In [7]:
# 問題 0.7 数値検証: 逆モード自動微分と2変量ラプラス求積の精度検証
import numpy as np
import scipy.integrate as integrate

# 1. 逆モード自動微分の数値検証
x_val = 1.2
t_val = 0.5
w1, w2 = 0.8, -0.4

# Forward Pass
a = w1 * x_val + w2
z = np.tanh(a)
diff = z - t_val
E = 0.5 * diff**2

# Backward Pass (Reverse-Mode AD)
bar_E = 1.0
bar_diff = bar_E * diff
bar_z = bar_diff * 1.0
bar_a = bar_z * (1.0 - z**2)
bar_w1 = bar_a * x_val
bar_w2 = bar_a * 1.0

# 中心差分 (有限差分)
eps = 1e-6
grad_w1_num = (0.5 * (np.tanh((w1 + eps) * x_val + w2) - t_val)**2 - 0.5 * (np.tanh((w1 - eps) * x_val + w2) - t_val)**2) / (2 * eps)
grad_w2_num = (0.5 * (np.tanh(w1 * x_val + (w2 + eps)) - t_val)**2 - 0.5 * (np.tanh(w1 * x_val + (w2 - eps)) - t_val)**2) / (2 * eps)

np.testing.assert_allclose(bar_w1, grad_w1_num, atol=1e-5)
np.testing.assert_allclose(bar_w2, grad_w2_num, atol=1e-5)
print(f'AD grad w1: {bar_w1:.6f}, Num grad w1: {grad_w1_num:.6f}')
print(f'AD grad w2: {bar_w2:.6f}, Num grad w2: {grad_w2_num:.6f}')

# 2. 2次元ラプラス近似の数値検証
def E_func(w1, w2):
    return w1**2 + 2.0 * w2**2 + 0.3 * w1 * w2 + 0.02 * w1**4 + 0.01 * w2**4

# w* = (0, 0), E(w*) = 0
A = np.array([[2.0, 0.3], [0.3, 4.0]])
det_A = np.linalg.det(A)
W_dim = 2
Z_laplace = np.exp(0.0) * (2.0 * np.pi)**(W_dim / 2.0) / np.sqrt(det_A)
Z_exact, _ = integrate.dblquad(lambda w2, w1: np.exp(-E_func(w1, w2)), -5, 5, -5, 5)

rel_err = abs(Z_laplace - Z_exact) / Z_exact
print(f'Z (Laplace): {Z_laplace:.6f}, Z (Exact): {Z_exact:.6f}, Relative Error: {rel_err:.4%}')
assert rel_err < 0.05
print('問題 0.7 検証完了: リバースモードAD・ラプラス求積ともに高精度で理論値と一致しました。')

AD grad w1: 0.007103, Num grad w1: 0.007103
AD grad w2: 0.005919, Num grad w2: 0.005919
Z (Laplace): 2.234044, Z (Exact): 2.198242, Relative Error: 1.6287%
問題 0.7 検証完了: リバースモードAD・ラプラス求積ともに高精度で理論値と一致しました。
